In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from xgboost.callback import EarlyStopping

In [18]:
  df = pd.read_csv(r"/content/df_train_modified.csv")

In [19]:
df.columns.to_list()

['DriverId',
 'TeamName',
 'FullName',
 'Position',
 'ClassifiedPosition',
 'GridPosition',
 'Time',
 'Status',
 'Points',
 'Laps',
 'year',
 'gp',
 'form_of_race',
 'is_circuit_street',
 'Q1',
 'Q2',
 'Q3',
 'quali_position',
 'n_corners',
 'overtaking_difficulty',
 'grid_penalty',
 'gp_round',
 'Position_num',
 'Position_filled',
 'is_dnf',
 'Position_for_form',
 'driver_avg_pos_3',
 'driver_avg_pos_5',
 'driver_dnf_rate_5']

In [20]:
for q in ['Q1', 'Q2', 'Q3']:
    df[q] = pd.to_timedelta(df[q]).dt.total_seconds()


In [21]:
current_df_cols = df.columns.tolist()
cols_to_drop = [col for col in ['Position','Laps', 'ClassifiedPosition', 'Time', 'Status', 'Points',
                  'Position_filled', 'is_dnf', 'Position_for_form','form_of_race'] if col in current_df_cols]
df = df.drop(columns=cols_to_drop)

In [22]:
cat_cols = ['DriverId', 'TeamName', 'gp']
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))


In [23]:
X = df.drop(columns=['Position_num', 'FullName'])
y = df['Position_num']

# Drop rows where y contains NaN values
valid_indices = y.notna()
X = X[valid_indices]
y = y[valid_indices]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_list = []
importances = np.zeros(X.shape[1])

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]


In [24]:
#Wagi lat
weights = X['year'].map({2022: 0.5, 2023: 0.5, 2024: 0.5, 2025: 0.5, 2026: 5.0})

In [25]:
model = XGBRegressor(
        max_depth=5,
        n_estimators=500,
        learning_rate=0.03,
        reg_alpha=0.5,
        reg_lambda=5,
        subsample=0.8,
        colsample_bytree=0.7,
        random_state=42)


In [26]:
model.fit(
        X_train, y_train,
        sample_weight=weights.iloc[train_idx],
        eval_set=[(X_val, y_val)],
        verbose=False
    )
y_pred = model.predict(X_val)
mae = mean_absolute_error(y_val, y_pred)
mae_list.append(mae)
importances += model.feature_importances_

print(f"Średni MAE (cross-val): {np.mean(mae_list):.3f}")
feat_imp = pd.Series(importances / kf.n_splits, index=X.columns).sort_values(ascending=False)
print("Top feature importance:")
print(feat_imp)

In [27]:
#finalny model

model = XGBRegressor(max_depth=5, n_estimators=500, learning_rate=0.03,
                     reg_alpha=0.5, reg_lambda=5, subsample=0.8,
                     colsample_bytree=0.7, random_state=42)
model.fit(X, y, sample_weight=weights, verbose=False)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [28]:
from sklearn.metrics import accuracy_score, roc_auc_score

#  kierowca jest w Top 5
y_val_binary_5 = (y_val <= 5).astype(int)
y_pred_binary_5 = (y_pred <= 5).astype(int)

# Accuracy dla Top 5
accuracy_5 = accuracy_score(y_val_binary_5, y_pred_binary_5)
print(f"Accuracy (Top 5): {accuracy_5:.3f}")

# ROC AUC dla Top 5
y_scores = -y_pred
roc_auc_5 = roc_auc_score(y_val_binary_5, y_scores)
print(f"ROC AUC (Top 5): {roc_auc_5:.3f}")

Accuracy (Top 5): 0.829
ROC AUC (Top 5): 0.893


In [29]:
# recznie wprowadzane dane dla miami
miami_data = pd.DataFrame({
    'Driver':               ['Antonelli','Russell','Leclerc','Hamilton','Norris',
                             'Piastri','Bearman','Gasly','Verstappen','Lawson',
                             'Lindblad','Hadjar','Bortoleto','Sainz','Ocon',
                             'Colapinto','Hulkenberg','Albon','Bottas','Perez',
                             'Alonso','Stroll'],
    'DriverId':             ['antonelli','russell','leclerc','hamilton','norris',
                             'piastri','bearman','gasly','max_verstappen','lawson',
                             'arvid_lindblad','hadjar','bortoleto','sainz','ocon',
                             'colapinto','hulkenberg','albon','bottas','perez',
                             'alonso','stroll'],
    'TeamName':             ['Mercedes','Mercedes','Ferrari','Ferrari','McLaren',
                             'McLaren','Haas F1 Team','Alpine','Red Bull Racing','Racing Bulls',
                             'Racing Bulls','Red Bull Racing','Audi','Williams','Haas F1 Team',
                             'Alpine','Audi','Williams','Cadillac','Cadillac',
                             'Aston Martin','Aston Martin'],
    'quali_position':       [1, 2, 4, 6, 5, 3, 12, 7, 11, 14,
                             10, 8, 13, 16, 15, 17, 18, 19, 20, 21,
                             22, 9],
    'GridPosition':         [1, 2, 4, 6, 5, 3, 12, 7, 11, 14,
                             10, 8, 13, 16, 15, 17, 18, 19, 20, 21,
                             22, 9],
    'driver_avg_pos_3':     [1.333, 10.0, 3.333, 10.0, 2.333, 4.333,
                             14.0, 13.333, 13.333, 7.667, 14.667, 9.667,
                             11.333, 13.0, 11.667, 18.0, 11.333, 17.667,
                             18.667, 17.0, 16.0, 14.333],
    'driver_avg_pos_5':     [1.333, 10.0, 3.333, 10.0, 2.333, 4.333,
                             14.0, 13.333, 13.333, 7.667, 14.667, 9.667,
                             11.333, 13.0, 11.667, 18.0, 11.333, 17.667,
                             18.667, 17.0, 16.0, 14.333],
    'driver_dnf_rate_5':    [0.0, 0.0, 0.0, 0.0, 0.33, 0.33, 0.33, 0.0,
                             0.33, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                             0.0, 0.0, 0.0, 0.0, 0.0, 0.33, 0.67],
})

# Stale cechy toru Miami
miami_data['year'] = 2026
miami_data['gp'] = 'Miami Grand Prix'
miami_data['is_circuit_street'] = 1
miami_data['n_corners'] = 19
miami_data['overtaking_difficulty'] = 4
miami_data['gp_round'] = 4
miami_data['grid_penalty'] = 0
miami_data['Q1'] = np.nan
miami_data['Q2'] = np.nan
miami_data['Q3'] = np.nan

In [30]:
driver_names = miami_data['Driver'].copy()
team_names = miami_data['TeamName'].copy()

from sklearn.preprocessing import LabelEncoder

for col in ['DriverId', 'TeamName', 'gp']:
    le = LabelEncoder()
    miami_data[col] = le.fit_transform(miami_data[col].astype(str))

miami_X = miami_data[X.columns]

In [31]:
miami_data['pred'] = model.predict(miami_X)
miami_data['Driver'] = driver_names.values
miami_data['Team'] = team_names.values

In [32]:
#sortowanie i wynik
top5 = miami_data.sort_values('pred').head(5).reset_index(drop=True)
top5.index += 1
print("\nMIAMI GP 2026 — PREDICTED TOP 5")
print("=" * 45)
for i, r in top5.iterrows():
    print(f"  P{i}  {r['Driver']:<20s} ({r['Team']})")
print("=" * 45)


MIAMI GP 2026 — PREDICTED TOP 5
  P1  Piastri              (McLaren)
  P2  Leclerc              (Ferrari)
  P3  Antonelli            (Mercedes)
  P4  Russell              (Mercedes)
  P5  Norris               (McLaren)
